In [1]:
import torch # pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import cv2, os, json, gc, math
import numpy as np
import pandas as pd
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from Network.index import ModelNetwork
from utils.Files import *

In [ ]:
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

gc.collect()
print(torch.__version__)              # versão do PyTorch
print(torch.cuda.is_available())      # True se detectou a GPU
print(torch.cuda.get_device_name(0))  # nome da GPU
pd.set_option('display.max_columns', None)

# HISTÓRICO DE TREINAMENTO

In [ ]:
df = []

for bakup in os.listdir('Backup/'):
    path = f'Backup/{bakup}/info.json'

    with open(path, 'r', encoding='utf-8') as file:
        file = json.loads(file.read())
    
    info = {}
    for key, value in file.items():
        info.update(value) 

    df.append(info)


df = pd.DataFrame(df)
df

In [ ]:
df = []

for bakup in os.listdir('Backup/'):
    path = f'Backup/{bakup}/info.json'

    with open(path, 'r', encoding='utf-8') as file:
        info = json.loads(file.read())
    

    df.append(info)


df = pd.DataFrame(df)
df

In [ ]:
from utils.Plotter.index import Plotter


class VariationAnalysis:
    def __init__(self, df, variations=[], metric='test_iou'):
        self.df = df.copy()
        self.variations = variations
        self.results = [] 
        self.metric  = metric

    def update(self):
        df = self.df.copy()

        for col in df.columns:
            if df[col].dtype == object:
                df[col] = df[col].astype(str)

        for variation in self.variations:
            const_cols = [col for col in self.variations if col != variation and col in df.columns]
            
            if const_cols:
                grouped = df.groupby(const_cols)
                
                for group_name, group in grouped:
                    if group[variation].nunique() > 1:
                        if not isinstance(group_name, tuple):
                            group_name = (group_name,)
                        
                        const_dict = dict(zip(const_cols, group_name))
                        
                        self.results.append({'variation': variation, 'fixed': const_dict, 'data': group.copy()})
            else:
                if df[variation].nunique() > 1:
                    self.results.append({'variation': variation, 'fixed': {}, 'data': df.copy()})
                    

    def format(self, df, variation): 
        other_cols = [c for c in df.columns if c != variation and c != self.metric]
        agg_funcs  = {self.metric: 'mean'}
        
        for col in other_cols:
            agg_funcs[col] = 'first'
            
        return df.groupby([variation], as_index=False).agg(agg_funcs)

    def plot(self):
        for res in self.results:
            variation = res['variation']
            df_group  = res['data']
            print('target:', variation, '| fixed:', res['fixed'])
            
            if len(df_group) == 0:
                continue
            
            df_table = df_group[self.variations + [self.metric]].copy()
            df_table[self.metric] = df_table[self.metric].round(4)
            df_str = df_table.astype(str)

            plt.figure(figsize=(20, 5))
            plt.subplot(1, 2, 1) 
            plt.axis('off')
            plt.title(f'Variation in: {variation}')
            
            table = plt.table(cellText=df_str.values, colLabels=df_table.columns, loc='center', cellLoc='center')
            table.auto_set_font_size(False)
            table.set_fontsize(10)
            table.scale(1, 1.5)
            
            plt.subplot(1, 2, 2)
            df_formatted = self.format(df_group, variation)
            Plotter({key: value for key, value in zip(df_formatted[variation], df_formatted[self.metric])})
            plt.show()


variations = ['loss', 'model_network', 'lr', 'img_size']
analysis   = VariationAnalysis(df, variations, metric='test_iou')
analysis.update()
analysis.plot()

# IMPORTANDO DADOS

In [ ]:
import torch.nn.functional as F
from torch import amp
from Network.index import ModelNetwork

In [ ]:
class Loader:   
    def __init__(self, index=-1):
        indexes = sorted([int(path.split('_')[-1]) for path in os.listdir('Backup/')]) if len('Backup/') > 0 else [0]
        self.index = index if index > 0 else indexes[-1]
        self.path  = f'Backup/model_{self.index}'
        info = json.load(open(f'{self.path}/info.json', 'r', encoding='utf-8'))

        self.network = ModelNetwork(**info['model'])  
        model_data   = torch.load(f'{self.path}/data.pth', map_location=self.network.device)

        self.history   = model_data['history']
        self.timestamp = model_data['timestamp']
        
        self.network.model.load_state_dict(model_data['model'])
        self.network.optimizer.load_state_dict(model_data['optimizer'])

        print(f'model {self.index} loaded, trained in {model_data["timestamp"]}')
        print(json.dumps(info, indent=4))

    def plot(self, save=False):
        history = self.history
        plt.figure(figsize=(17, 5))
        plt.subplot(1, 2, 1)
        plt.plot([h.get('train_loss') for h in history], 'y', label='Training loss')
        plt.plot([h.get('val_loss') for h in history], 'r', label='Validation loss')
        plt.title('Training and validation loss')
        plt.xlabel('epoch'), plt.ylabel('loss')
        plt.legend(), plt.grid()

        plt.subplot(1, 2, 2)
        plt.plot([h.get('train_iou') for h in history], 'y', label='Training IoU')
        plt.plot([h.get('val_iou')   for h in history], 'r',  label='Validation IoU')
        plt.title('Training and validation IoU')
        plt.xlabel('epoch'), plt.ylabel('IoU')
        plt.legend(), plt.grid(), plt.gca().set_ylim(0, 1), plt.yticks([c/10 for c in range(11)])
        
        if not save:
            return plt.show()

        plt.savefig(f'{self.path}/train.png', dpi=300, bbox_inches='tight', facecolor='white')
        plt.show()
        plt.close()

loader = Loader(-1)
loader.plot(True)

In [ ]:
network = loader.network
network.model

# DOWNLOAD DOS DADOS

In [ ]:
from torch.utils.data import TensorDataset, DataLoader
from torchmetrics.classification import MulticlassJaccardIndex
from torchmetrics.classification import BinaryJaccardIndex

In [ ]:
def getLoader(xData, yData, batch, shuffle=False):
    xTensor = torch.from_numpy(xData).float()
    yTensor = torch.from_numpy(yData).long() if network.multiclass else torch.from_numpy(yData).float()

    if xTensor.ndim == 4:
        xTensor = xTensor.unsqueeze(1)
    
    if yTensor.ndim == 4: 
        yTensor = yTensor.unsqueeze(1)
    
    dataset = TensorDataset(xTensor, yTensor)
    return DataLoader(dataset, batch_size=batch, shuffle=shuffle, pin_memory=True)


xTest = np.array([np.load(path) for path in getFiles('Database/Processed/test/images')])
yTest = np.array([np.load(path) for path in getFiles('Database/Processed/test/masks')])
test_loader = getLoader(xTest,  yTest, 1, shuffle=False)

In [ ]:
class_iou = MulticlassJaccardIndex(num_classes=network.classes, average=None).to(network.device)
network.model.eval()
network.iou.reset()
yModel = []

with torch.no_grad():
    for (imgs, masks) in test_loader:
        imgs, masks = imgs.to(network.device), masks.to(network.device)
        logits = network.model(imgs)
        
        if network.multiclass:
            preds  = torch.argmax(logits, dim=1)
            target = masks.squeeze(1) if masks.dim() == 5 else masks
        else:
            preds  = (torch.sigmoid(logits) > 0.5).int()
            target = masks.squeeze(1).int() if masks.dim() == 5 else masks.int()
        
        network.iou.update(preds, target)
        yModel.append(preds.cpu().numpy()[0])
        class_iou.update(preds, target)

test_iou = network.iou.compute().item()
print('test mean iou:', test_iou)

iouData = {f'class_{i}': score.item() for i, score in enumerate(class_iou.compute())}
print(iouData)

plt.figure(figsize=(10, 5))
plt.bar(iouData.keys(), iouData.values())
plt.ylabel('IoU')
plt.title('IoU Variation per Class')
plt.grid(alpha=0.3)
plt.savefig(f'{loader.path}/iou_classes.png')
plt.show()

In [ ]:
def plotTest(img, mask, pred, alpha=0.5, savePath=None):
    mid = img.shape[0] // 2
    imgSlice, maskSlice, predSlice = img[mid], mask[mid], pred[mid]
    
    imgNorm = cv2.normalize(imgSlice, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    imgRgb  = cv2.cvtColor(imgNorm, cv2.COLOR_GRAY2RGB)
    
    def applyColor(targetMask):
        overlay = imgRgb.copy()
        for cls in np.unique(targetMask):
            if cls > 0: 
                np.random.seed(int(cls))
                color = (np.array(plt.cm.hsv(np.random.random())[:3]) * 255).astype(int).tolist()
                overlay[targetMask == cls] = color
        return cv2.addWeighted(overlay, alpha, imgRgb, 1 - alpha, 0)

    plt.figure(figsize=(18, 5))
    plt.subplot(1, 3, 1)
    plt.imshow(imgRgb)
    plt.title('Original Image')
    plt.axis('off')
    
    plt.subplot(1, 3, 2)
    plt.imshow(applyColor(maskSlice))
    plt.title('Ground Truth')
    plt.axis('off')
    
    plt.subplot(1, 3, 3)
    plt.imshow(applyColor(predSlice))
    plt.title('Model Prediction')
    plt.axis('off')
    plt.tight_layout()

    if savePath: plt.savefig(savePath, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    plt.close()


os.makedirs(f'{loader.path}/predictions', exist_ok=True)
for i in range(min(len(xTest), 7)):
    path = f'{loader.path}/predictions/pred_{i}.png'
    plotTest(xTest[i], yTest[i], yModel[i], savePath=path)